## Imports

In [2]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [3]:
import numpy as np

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [5]:
from ucimlrepo import fetch_ucirepo

In [6]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [7]:
X = X.fillna(value=0)

In [8]:
y

,num
0,0
1,2
2,1
3,0
4,0
...,...
298,1
299,2
300,3
301,1


In [9]:
y = y.copy()
y.loc[y["num"] > 0, "num"] = 1

In [10]:
y

,num
0,0
1,1
2,1
3,0
4,0
...,...
298,1
299,1
300,1
301,1


In [11]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y)

In [12]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[82 69]


In [13]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[82 70]


In [14]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [15]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [16]:
y_train.dtype

torch.int64

In [17]:
x_train = x_train.type(torch.float64)
x_test = x_test.type(torch.float64)

In [18]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 4, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [19]:
y_trainset.dtype

torch.int64

## Model & Training

In [20]:
2**13

8192

In [21]:
model = nft.h_ANFIS(
    input_size = 13,
    num_mfs = 40,
    outputs = 2,
    #membership_function=nft.Gaussian_MF,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float64
)

#model.init_premises(x_trainset)

#model.init_consequents(x_trainset, y_trainset)

In [22]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=50, 
    delta=0.01
)

In [23]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=1000,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [24]:
trainer(model, loader, verbose=True)

Epoch:    1/1000 - loss: 0.974338 - validation loss: 1.063101
Epoch:    2/1000 - loss: 0.957909 - validation loss: 1.058151
Epoch:    3/1000 - loss: 0.938391 - validation loss: 1.052303
Epoch:    4/1000 - loss: 0.921922 - validation loss: 1.049007
Epoch:    5/1000 - loss: 0.903449 - validation loss: 1.043822
Epoch:    6/1000 - loss: 0.884889 - validation loss: 1.038224
Epoch:    7/1000 - loss: 0.865231 - validation loss: 1.032736
Epoch:    8/1000 - loss: 0.849510 - validation loss: 1.027797
Epoch:    9/1000 - loss: 0.828961 - validation loss: 1.021885
Epoch:   10/1000 - loss: 0.812075 - validation loss: 1.017436
Epoch:   11/1000 - loss: 0.796713 - validation loss: 1.012304
Epoch:   12/1000 - loss: 0.779604 - validation loss: 1.005931
Epoch:   13/1000 - loss: 0.761540 - validation loss: 1.000204
Epoch:   14/1000 - loss: 0.746851 - validation loss: 0.997523
Epoch:   15/1000 - loss: 0.731778 - validation loss: 0.991411
Epoch:   16/1000 - loss: 0.714950 - validation loss: 0.985657
Epoch:  

In [25]:
train_measures = nft.get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.8807947019867549
Precision: 0.8813301899086591
Recall: 0.8807947019867549
F1: 0.8804346291249523
Confusion Matrix: [[75  7]
 [11 58]]


In [26]:
test_measures = nft.get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.7631578947368421
Precision: 0.7633261064593302
Recall: 0.7631578947368421
F1: 0.7620350261078509
Confusion Matrix: [[67 15]
 [21 49]]
